# 差分発現タンパク質の基本可視化

**対応記事**: [article-08b-differential-clustering.md](../blog/article-08b-differential-clustering.md) — 差分発現タンパク質の可視化  
**実行順序**: 8b番目  
**所要時間**: 約8分

---

## このNotebookで行うこと

前回（notebook_08a）で特定した1,055個の有意差タンパク質を使って、階層的クラスタリングと主成分分析（PCA）を実行します。有意差タンパク質だけに絞ることで、Normal/Tumor群の分離パターンがより明瞭になることを確認します。

- 有意差タンパク質の抽出と整理
- 階層的クラスタリングによるヒートマップ
- PCA解析による次元削減と群分離の可視化
- 95%信頼楕円による群分散の定量的評価

**⚠️ 注意**: このNotebookを実行する前に、notebook_08aの実行が完了している必要があります。

## 前提条件

- [notebook_08a_differential_volcano.ipynb](./notebook_08a_differential_volcano.ipynb) が完了していること
- 差分発現タンパク質データ（`differential_proteins.csv`）が生成済み
- 前処理済みデータが利用可能であること
- Python環境が適切に設定されていること

## 1. ライブラリと設定（基本）

In [ ]:
import numpy as np           # 数値計算ライブラリ（配列操作・数学関数に使用）
import pandas as pd          # データフレーム操作ライブラリ（表形式データの読み込み・加工に使用）
import matplotlib.pyplot as plt  # グラフ描画ライブラリ（PCAプロット等の作成に使用）
from matplotlib.patches import Ellipse  # 楕円パッチ（PCAプロットに95%信頼楕円を描画するため）
import matplotlib.transforms as transforms  # 座標変換（楕円の回転・拡縮・移動に使用）
import seaborn as sns        # 統計的可視化ライブラリ（ヒートマップ等の高水準プロットに使用）
from sklearn.decomposition import PCA  # 主成分分析（PCA）クラス（次元削減による群分離の可視化に使用）
import os                   # OS操作ライブラリ（ディレクトリ作成に使用）

# Jupyter notebook での図のインライン表示設定
%matplotlib inline

In [ ]:
# --- パス ---
RESULTS  = "../results"              # 解析結果の保存先ディレクトリへのパス
FIG_DIR  = f"{RESULTS}/figures"      # 図の保存先ディレクトリへのパス
TABLE_DIR = f"{RESULTS}/tables"      # テーブル（CSV等）の保存先ディレクトリへのパス

# ディレクトリが存在しない場合は作成
os.makedirs(FIG_DIR, exist_ok=True)
os.makedirs(TABLE_DIR, exist_ok=True)

print(f"結果保存先: {RESULTS}")
print(f"図保存先: {FIG_DIR}")
print(f"テーブル保存先: {TABLE_DIR}")

In [ ]:
# --- カラー ---
NORMAL, TUMOR = "#3498DB", "#E74C3C"  # Normal群を青、Tumor群を赤で表示する色コード
UP, DOWN, NS  = "#E74C3C", "#3498DB", "#CCCCCC"  # 上昇(赤)・低下(青)・非有意(灰)の色コード
# 条件名から色への対応辞書（グラフ描画時に群ごとの色を自動で割り当てるため）
COND_MAP = {"Normal": NORMAL, "Tumor": TUMOR}

print(f"色設定:")
print(f"Normal群: {NORMAL}, Tumor群: {TUMOR}")
print(f"Up-regulated: {UP}, Down-regulated: {DOWN}")

## 2. データ読み込み

In [ ]:
# 前処理済みタンパク質発現データを読み込む（行=タンパク質、列=サンプル、値=log2発現量）
df = pd.read_csv(f"{RESULTS}/preprocessed_data.csv", index_col=0)
# サンプル情報（各サンプルがNormalかTumorか）を読み込む
sample_info = pd.read_csv(f"{RESULTS}/sample_info.csv")
# サンプル名をインデックスにして条件列だけを取り出す（後でサンプルから条件を引くための辞書的Series）
conditions = sample_info.set_index("Sample")["Condition"]
# 前回の差分発現解析結果を読み込む
result_df = pd.read_csv(f"{TABLE_DIR}/differential_proteins.csv")

print("データ読み込み完了:")
print(f"タンパク質発現データ: {df.shape}")
print(f"サンプル情報: {sample_info.shape}")
print(f"差分発現解析結果: {result_df.shape}")

In [ ]:
# 有意差タンパク質の数を確認
n_up = (result_df["Significant"] == "Up").sum()
n_down = (result_df["Significant"] == "Down").sum()
n_ns = (result_df["Significant"] == "NS").sum()
n_significant = n_up + n_down

print(f"差分発現解析結果:")
print(f"Up-regulated: {n_up}個")
print(f"Down-regulated: {n_down}個")
print(f"Non-significant: {n_ns}個")
print(f"有意差タンパク質合計: {n_significant}個")

## 3. 信頼楕円のヘルパー関数

**【95%信頼楕円とは？】**

- **ひとことで**: データ点の95%が含まれる楕円領域を描画する関数
- **定義**: 2次元データの共分散構造に基づいて楕円の形状・向き・サイズを決定
- **どんなとき使う**: PCAプロットで各群のデータ分散範囲を視覚的に示すとき
- **読み方**: 楕円が重複しなければ群がよく分離している証拠
- **参考**: 多変量統計の標準的手法

In [ ]:
def confidence_ellipse(x, y, ax, n_std=2.0, **kwargs):
    """95% 信頼楕円を描画する。
    
    Parameters
    ----------
    x : array-like
        x座標のデータ
    y : array-like
        y座標のデータ
    ax : matplotlib.axes.Axes
        描画先の軸オブジェクト
    n_std : float, default=2.0
        標準偏差の倍数（2.0で約95%信頼区間）
    **kwargs
        楕円の描画オプション（色、透明度等）
    
    Returns
    -------
    matplotlib.patches.Ellipse
        描画された楕円オブジェクト
    """
    # データ点が2個未満では共分散を計算できないため、描画をスキップする
    if len(x) < 2:
        return
    # xとyの共分散行列を計算する（2x2行列: 分散と共分散を含む）
    cov = np.cov(x, y)
    # ピアソン相関係数を計算する（xとyの線形関係の強さ。-1〜+1の範囲）
    pearson = cov[0, 1] / np.sqrt(cov[0, 0] * cov[1, 1])
    # 楕円の横半径を計算する（相関が高いほど横長になる）
    ell_rx = np.sqrt(1 + pearson)
    # 楕円の縦半径を計算する（相関が高いほど縦短になる）
    ell_ry = np.sqrt(1 - pearson)
    # 原点中心の楕円オブジェクトを作成する（**kwargsで色や透明度を受け取る）
    ellipse = Ellipse((0, 0), width=ell_rx * 2, height=ell_ry * 2, **kwargs)
    # アフィン変換を組み合わせて楕円をデータの分布に合わせる
    transf = (transforms.Affine2D()
              .rotate_deg(45)  # 45度回転（共分散の主軸方向に合わせる）
              # n_std倍の標準偏差に合わせてスケールする（n_std=2で約95%信頼区間）
              .scale(np.sqrt(cov[0, 0]) * n_std, np.sqrt(cov[1, 1]) * n_std)
              # データの重心位置に移動する
              .translate(np.mean(x), np.mean(y)))
    # 変換を楕円に適用する（ax.transDataでデータ座標系に変換）
    ellipse.set_transform(transf + ax.transData)
    # 楕円をAxesに追加して描画する
    return ax.add_patch(ellipse)

print("信頼楕円描画関数を定義しました")

## 4. 有意差タンパク質の抽出と前処理

In [ ]:
# --- 有意差タンパク質の抽出 ---
# 検定結果から有意差あり（UpまたはDown）のタンパク質だけを抽出する
sig_df = result_df[result_df["Significant"] != "NS"]
# 元の発現量データから有意差タンパク質の行だけを取り出す
sig_data = df.loc[df.index.isin(sig_df["Protein"])]
# 各タンパク質の発現変化方向（Log2FC）をSeriesとして取得する（色分けに使用）
direction = sig_df.set_index("Protein")["Log2FC"]

print(f"有意差タンパク質の抽出完了:")
print(f"元データ: {len(df)}個のタンパク質")
print(f"有意差タンパク質: {len(sig_data)}個のタンパク質")
print(f"抽出率: {len(sig_data)/len(df)*100:.1f}%")

# 抽出された有意差タンパク質の統計
print(f"\n有意差タンパク質の詳細:")
print(f"形状: {sig_data.shape}")
print(f"欠損値の数: {sig_data.isnull().sum().sum()}")

In [ ]:
# 各サンプルの条件（Normal/Tumor）に応じた色をSeriesとして作成する（ヒートマップの行カラーバーに使用）
sample_colors  = conditions.reindex(df.columns).map(COND_MAP)
# 各タンパク質の発現変化方向に応じた色をSeriesとして作成する（ヒートマップの列カラーバーに使用）
# Log2FC > 0 なら赤（Up: 腫瘍で増加）、それ以外は青（Down: 腫瘍で減少）
protein_colors = pd.Series(
    sig_data.index.map(lambda p: UP if direction.get(p, 0) > 0 else DOWN),
    index=sig_data.index, name="Expression"
)

print(f"カラーマッピング作成完了:")
print(f"サンプル色数: {len(sample_colors)}")
print(f"タンパク質色数: {len(protein_colors)}")
print(f"\nサンプル色の分布:")
print(sample_colors.value_counts())
print(f"\nタンパク質色の分布:")
print(protein_colors.value_counts())

## 5. 階層的クラスタリングによるヒートマップ

In [ ]:
# --- Figure 2a: ヒートマップ ---
print("階層的クラスタリング付きヒートマップを作成中...")

# 階層的クラスタリング付きヒートマップを作成する
g = sns.clustermap(
    sig_data.T,              # データを転置する（行=サンプル、列=タンパク質にする）
    method="ward",           # Ward法でクラスタリングする（分散を最小化する手法）
    cmap="RdBu_r",           # カラーマップ: 赤=高発現、青=低発現（反転版）
    z_score=1,               # 列方向（タンパク質ごと）に標準化する（平均0、標準偏差1に変換）
    row_colors=sample_colors,   # 行（サンプル）の横にNormal/Tumorの色バーを表示する
    col_colors=protein_colors,  # 列（タンパク質）の上にUp/Downの色バーを表示する
    xticklabels=False,       # x軸のタンパク質名ラベルを非表示にする（数が多すぎて読めないため）
    yticklabels=True,        # y軸のサンプル名ラベルを表示する
    figsize=(10, 8),         # 図のサイズを横10×縦8インチに設定する
    vmin=-3, vmax=3,         # 色の範囲を-3〜+3に制限する（外れ値による色の偏りを防ぐ）
)
# y軸のサンプル名のフォントサイズを小さくして見やすくする
g.ax_heatmap.set_yticklabels(g.ax_heatmap.get_yticklabels(), fontsize=7)
# ヒートマップをPNGファイルとして保存する
g.savefig(f"{FIG_DIR}/fig2a_heatmap_all.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"ヒートマップを保存しました: {FIG_DIR}/fig2a_heatmap_all.png")

## 6. 主成分分析（PCA）

In [ ]:
# --- Figure 2b: PCA ---
print("主成分分析（PCA）を実行中...")

# 主成分分析（PCA）オブジェクトを作成する（2次元に次元削減する）
pca = PCA(n_components=2)
# 有意差タンパク質の発現データでPCAを実行する（転置して行=サンプルにする）
# scores: 各サンプルのPC1, PC2座標を格納した配列
scores = pca.fit_transform(sig_data.T)

print(f"PCA解析完了:")
print(f"入力データ形状: {sig_data.T.shape}")
print(f"PCAスコア形状: {scores.shape}")
print(f"第1主成分の寄与率: {pca.explained_variance_ratio_[0]*100:.1f}%")
print(f"第2主成分の寄与率: {pca.explained_variance_ratio_[1]*100:.1f}%")
print(f"累積寄与率: {pca.explained_variance_ratio_.sum()*100:.1f}%")

In [ ]:
# PCAプロット用の図と軸を作成する
fig, ax = plt.subplots(figsize=(8, 6))

# Normal群とTumor群をそれぞれ異なる色でプロットする
for cond, color in [("Normal", NORMAL), ("Tumor", TUMOR)]:
    # 各サンプルが現在の条件に該当するかのブールマスクを作成する
    m = (conditions.reindex(df.columns) == cond).values
    # 散布図を描画する（s=80: 点のサイズ、alpha=0.8: やや透明、edgecolors="white": 白い縁取り）
    ax.scatter(scores[m, 0], scores[m, 1], c=color, s=80, alpha=0.8,
              label=cond, edgecolors="white")
    # 95%信頼楕円を描画する（群の分布範囲を視覚的に示す）
    confidence_ellipse(scores[m, 0], scores[m, 1], ax,
                       facecolor=color, alpha=0.15, edgecolor=color, lw=1.5)
    print(f"{cond}群のサンプル数: {m.sum()}個")

# x軸ラベル: 第1主成分と寄与率（全分散のうちPC1が説明する割合）を表示する
ax.set_xlabel(f"Component 1 ({pca.explained_variance_ratio_[0]*100:.1f}%)")
# y軸ラベル: 第2主成分と寄与率を表示する
ax.set_ylabel(f"Component 2 ({pca.explained_variance_ratio_[1]*100:.1f}%)")
# グラフタイトル: 全有意差タンパク質を使ったPCA
ax.set_title("PCA using All Differentially Abundant Proteins")
# 凡例を表示する（frameon=Falseで枠線を非表示にする）
ax.legend(frameon=False)
# 上辺と右辺の枠線を非表示にする
ax.spines[["top", "right"]].set_visible(False)
# 薄い破線のグリッドを表示する（データの位置を読み取りやすくする）
ax.grid(True, alpha=0.3, ls="--")

# PCAプロットをPNGファイルとして保存する
fig.savefig(f"{FIG_DIR}/fig2b_pca_all.png", dpi=150, bbox_inches="tight")
plt.show()

print(f"PCAプロットを保存しました: {FIG_DIR}/fig2b_pca_all.png")

## 7. 群分離の定量的評価

In [ ]:
# PCAスコアを使った群分離の定量的評価
normal_mask = (conditions.reindex(df.columns) == "Normal").values
tumor_mask = (conditions.reindex(df.columns) == "Tumor").values

# 各群の重心を計算
normal_centroid = scores[normal_mask].mean(axis=0)
tumor_centroid = scores[tumor_mask].mean(axis=0)

# 群間距離（ユークリッド距離）
between_group_distance = np.linalg.norm(normal_centroid - tumor_centroid)

# 群内分散
normal_var = np.var(scores[normal_mask], axis=0).sum()
tumor_var = np.var(scores[tumor_mask], axis=0).sum()
within_group_variance = (normal_var + tumor_var) / 2

# 分離度指標（群間距離 / 群内分散の平方根）
separation_ratio = between_group_distance / np.sqrt(within_group_variance)

print(f"【PCA群分離の定量的評価】")
print(f"Normal群重心: PC1={normal_centroid[0]:.2f}, PC2={normal_centroid[1]:.2f}")
print(f"Tumor群重心:  PC1={tumor_centroid[0]:.2f}, PC2={tumor_centroid[1]:.2f}")
print(f"群間距離: {between_group_distance:.2f}")
print(f"群内分散: Normal={normal_var:.2f}, Tumor={tumor_var:.2f}")
print(f"分離度指標: {separation_ratio:.2f} （値が大きいほど良好な分離）")

In [ ]:
# PCA負荷量（どのタンパク質が主成分に寄与しているか）の解析
loadings = pca.components_.T  # 負荷量行列（タンパク質×主成分）
loading_df = pd.DataFrame(
    loadings,
    index=sig_data.index,
    columns=["PC1", "PC2"]
)

# PC1に最も寄与するタンパク質（正・負方向）
pc1_top_positive = loading_df.nlargest(5, "PC1")["PC1"]
pc1_top_negative = loading_df.nsmallest(5, "PC1")["PC1"]

# PC2に最も寄与するタンパク質（正・負方向）
pc2_top_positive = loading_df.nlargest(5, "PC2")["PC2"]
pc2_top_negative = loading_df.nsmallest(5, "PC2")["PC2"]

print(f"\n【PCA負荷量解析】")
print(f"PC1正方向（Normal群側）に寄与するタンパク質 Top5:")
print(pc1_top_positive)
print(f"\nPC1負方向（Tumor群側）に寄与するタンパク質 Top5:")
print(pc1_top_negative)

print(f"\nPC2正方向に寄与するタンパク質 Top5:")
print(pc2_top_positive)
print(f"\nPC2負方向に寄与するタンパク質 Top5:")
print(pc2_top_negative)

## 8. 結果の保存と統計サマリー

In [ ]:
# PCA結果を保存する
pca_results = pd.DataFrame(
    scores,
    index=df.columns,
    columns=["PC1", "PC2"]
)
# サンプル情報とマージ
pca_with_conditions = pca_results.reset_index().merge(
    sample_info, left_on="index", right_on="Sample"
)

# PCA結果をCSVファイルに保存
pca_output_file = f"{TABLE_DIR}/pca_scores_significant_proteins.csv"
pca_with_conditions.to_csv(pca_output_file, index=False)
print(f"PCAスコアを保存しました: {pca_output_file}")

# 負荷量データも保存
loading_output_file = f"{TABLE_DIR}/pca_loadings_significant_proteins.csv"
loading_df.to_csv(loading_output_file)
print(f"PCA負荷量を保存しました: {loading_output_file}")

# 有意差タンパク質のデータも保存
sig_data_output_file = f"{TABLE_DIR}/significant_proteins_data.csv"
sig_data.to_csv(sig_data_output_file)
print(f"有意差タンパク質データを保存しました: {sig_data_output_file}")

## まとめ

このNotebookでは以下の解析を実行しました：

1. **有意差タンパク質の抽出**: 1,055個の有意差タンパク質を元データから抽出
2. **階層的クラスタリング**: Ward法による発現パターンの類似性に基づくグルーピング
3. **PCA解析**: 2次元空間での群分離の可視化と定量的評価
4. **95%信頼楕円**: 各群のデータ分散範囲の統計的評価

**主要な発見:**
- **明瞭な群分離**: ヒートマップとPCAの両方でNormal/Tumor群が明確に分離
- **効果的な絞り込み**: 有意差タンパク質に絞ることでノイズが除去され分離が鮮明化
- **主成分の意義**: 第1・第2主成分で群間差異を効果的に捉えることを確認
- **分離度指標**: 定量的評価により良好な分離性能を確認

**技術的成果:**
- 有意差タンパク質が大腸がんの分子的特徴を適切に反映
- 階層的クラスタリングによる発現パターンの構造化
- PCA負荷量解析による重要タンパク質の特定

**次のステップ:**
これらの有意差タンパク質の中から最も変化の大きいタンパク質群を段階的に選択し、バイオマーカーパネルとしての実用性を評価します。

---

## Navigation

⬅️ **前回**: [notebook_08a_differential_volcano.ipynb](./notebook_08a_differential_volcano.ipynb) — Volcanoプロット解析  
➡️ **次回**: [notebook_08c_differential_topn_analysis.ipynb](./notebook_08c_differential_topn_analysis.ipynb) — トップN解析

---

*このNotebookは [article-08b-differential-clustering.md](../blog/article-08b-differential-clustering.md) に対応しています。*